# Usage Tracking - Monitor Token Consumption

## Purpose
Learn how to track token usage and costs for your agent applications. Understanding token consumption is critical for optimizing performance, managing budgets, and identifying expensive operations.

## Key Concepts
- **Usage Statistics**: Token metrics captured per agent execution
- **context_wrapper.usage**: Access point for usage data
- **Token Types**: Input tokens (prompt), output tokens (response), total tokens
- **Requests**: Number of API calls made during execution

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
model_id = "openai.gpt-5.5"

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Import Libraries

In [ ]:
import asyncio

from agents import Agent, Runner, function_tool

## Step 1: Define Tools

Create simple function tools for a tour assistance agent:

In [ ]:
@function_tool        
def weather_tool(city: str) -> str:
    """Returns weather of a city."""
    return f"Weather is sunny in {city}."

@function_tool        
def history_tool(city: str) -> str:
    """Return history of a city."""
    return f"the{city} is very historical with great and old monuments."

## Step 2: Create Agent with Tools

Build a tour assistance agent that uses both tools:

In [ ]:
agent = Agent(
    name="Tour Assistance",
    instructions="Provides information about cities to the tourists.",
    tools=[weather_tool,history_tool],
    model=model_id,
)

## Step 3: Track Usage for First Query

Run the agent and access usage statistics via `result.context_wrapper.usage`:

**Usage Metrics**:
- **requests**: Number of API calls made
- **input_tokens**: Tokens in the prompt (instructions + conversation + tool definitions)
- **output_tokens**: Tokens in the model's response
- **total_tokens**: Sum of input and output tokens

🔍 **Watch**: Tool calls increase request count and token usage!

In [ ]:
result = await Runner.run(agent,"How is weather in Nice?")
print(result.final_output)

usage = result.context_wrapper.usage

print("Requests:", usage.requests)
print("Input tokens:", usage.input_tokens)
print("Output tokens:", usage.output_tokens)
print("Total tokens:", usage.total_tokens)

## Step 4: Compare Usage Across Queries

Run a different query and compare token usage:

⚡ **Note**: Different queries may use different tools and produce different response lengths, affecting token counts.

In [ ]:
result = await Runner.run(agent,"Tell me history of Nice")
print(result.final_output)

usage = result.context_wrapper.usage

print("Requests:", usage.requests)
print("Input tokens:", usage.input_tokens)
print("Output tokens:", usage.output_tokens)
print("Total tokens:", usage.total_tokens)

## 🎉 Congratulations!

You've completed the **Usage Tracking** notebook!